# Incremental Finetuning - Google Colab
**Stage-by-stage finetuning on T4 GPU**

Workflow per stage:
1. Load cumulative data (Stage 1 = 5K, Stage 2 = 10K, etc.)
2. Finetune Gemma 3:1B with LoRA
3. Generate outputs for ALL 50K with finetuned model
4. Evaluate and update Supabase with checkpoint columns

In [ ]:
# Cell 1: Install dependencies
!pip install unsloth supabase python-dotenv trl datasets -q
!pip install --upgrade transformers -q

In [ ]:
# Cell 2: Configuration - SET THESE VALUES!

# Supabase credentials
SUPABASE_URL = "YOUR_SUPABASE_URL"  # e.g., https://xxxxx.supabase.co
SUPABASE_KEY = "YOUR_SUPABASE_SERVICE_KEY"  # Use service role key

# Which stage to run (1-10)
STAGE = 1  # Change this for each stage!

# Training config
RECORDS_PER_STAGE = 5000  # 5K per stage
MAX_NEW_TOKENS = 256
EPOCHS = 1
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4

print(f"🎯 Stage {STAGE}: Will finetune on {STAGE * RECORDS_PER_STAGE} records")

In [ ]:
# Cell 3: Setup and imports
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

import torch
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True

import time
import json
from tqdm import tqdm
from supabase import create_client
from datasets import Dataset

# Connect to Supabase
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Cell 4: Load base model with LoRA for finetuning
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

print("Loading base model...")
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-1b-it-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

# Add LoRA adapters
print("Adding LoRA adapters...")
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
print("✅ Model ready for finetuning")

In [ ]:
# Cell 5: Fetch training data from Supabase (cumulative for this stage)
print(f"Fetching {STAGE * RECORDS_PER_STAGE} records for Stage {STAGE}...")

# Fetch cumulative data (stage 1 = first 5K, stage 2 = first 10K, etc.)
training_records = []
offset = 0
total_needed = STAGE * RECORDS_PER_STAGE

while len(training_records) < total_needed:
    result = supabase.table("modelcomp_50k") \
        .select("input, context, sevenb") \
        .order("id") \
        .range(offset, offset + 999) \
        .execute()
    
    if not result.data:
        break
    
    training_records.extend(result.data)
    offset += 1000
    print(f"  Fetched {len(training_records)}/{total_needed}...")

# Trim to exact amount
training_records = training_records[:total_needed]
print(f"✅ Got {len(training_records)} training records")

In [ ]:
# Cell 6: Format training data
def format_for_training(record):
    """Format record as chat for training"""
    instruction = record.get("input") or ""
    context = record.get("context") or ""
    output = record.get("sevenb") or ""  # Teacher output (Alpaca)
    
    if context:
        user_content = f"Context: {context}\n\nInstruction: {instruction}"
    else:
        user_content = instruction
    
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": output}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# Create dataset
print("Formatting training data...")
formatted_data = [format_for_training(r) for r in tqdm(training_records)]
train_dataset = Dataset.from_list(formatted_data)
print(f"✅ Dataset ready: {len(train_dataset)} examples")

In [ ]:
# Cell 7: Setup trainer and finetune
from trl import SFTTrainer
from transformers import TrainingArguments

print(f"\n🚀 Starting finetuning for Stage {STAGE}...")
print(f"   Records: {len(train_dataset)}")
print(f"   Epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} = {BATCH_SIZE * GRADIENT_ACCUMULATION}")

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=TrainingArguments(
        output_dir=f"./stage{STAGE}_output",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        warmup_steps=50,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=50,
        save_strategy="no",
        optim="adamw_8bit",
    ),
    dataset_text_field="text",
    max_seq_length=2048,
)

# Train!
train_start = time.time()
trainer.train()
train_time = time.time() - train_start

print(f"\n✅ Finetuning complete in {train_time/60:.1f} minutes")

In [ ]:
# Cell 8: Save finetuned model (optional - to Google Drive)
# Uncomment below to save to Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained(f"/content/drive/MyDrive/gemma_stage{STAGE}_lora")
# tokenizer.save_pretrained(f"/content/drive/MyDrive/gemma_stage{STAGE}_lora")

print("Model in memory, ready for generation")

In [ ]:
# Cell 9: Switch model to inference mode
model.eval()
device = "cuda"

def generate_output(instruction: str, context: str = "") -> tuple:
    """Generate output and return (text, latency)"""
    start_time = time.time()
    
    if context:
        user_content = f"Context: {context}\n\nInstruction: {instruction}"
    else:
        user_content = instruction
    
    messages = [{"role": "user", "content": user_content}]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    new_tokens = outputs[0][inputs.shape[1]:]
    output_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    
    latency = time.time() - start_time
    return output_text.strip(), latency

# Test generation
test_out, test_lat = generate_output("What is machine learning?")
print(f"Test ({test_lat:.2f}s): {test_out[:150]}...")

In [ ]:
# Cell 10: Generate outputs for ALL 50K with finetuned model
# This updates student_output_ckpt{STAGE} column

ckpt_output_col = f"student_output_ckpt{STAGE}"
ckpt_latency_col = f"latency_ckpt{STAGE}"

print(f"\n🔄 Generating outputs for Stage {STAGE}...")
print(f"   Output column: {ckpt_output_col}")
print(f"   Latency column: {ckpt_latency_col}")

# Check how many need generation
result = supabase.table("modelcomp_50k").select("id", count="exact").is_(ckpt_output_col, "null").execute()
remaining = result.count
print(f"   Records to generate: {remaining}")

BATCH_SIZE = 100
total_processed = 0
errors = []

while True:
    # Fetch batch without checkpoint output
    result = supabase.table("modelcomp_50k") \
        .select("id, input, context") \
        .is_(ckpt_output_col, "null") \
        .limit(BATCH_SIZE) \
        .execute()
    
    records = result.data
    if not records:
        print("\n✅ All records generated!")
        break
    
    for record in tqdm(records, desc=f"Stage {STAGE}"):
        try:
            record_id = record["id"]
            instruction = record["input"] or ""
            context = record["context"] or ""
            
            output, latency = generate_output(instruction, context)
            
            # Update checkpoint columns
            supabase.table("modelcomp_50k").update({
                ckpt_output_col: output,
                ckpt_latency_col: round(latency, 3)
            }).eq("id", record_id).execute()
            
            total_processed += 1
            
        except Exception as e:
            errors.append({"id": record_id, "error": str(e)})
    
    # Progress update
    remaining_result = supabase.table("modelcomp_50k").select("id", count="exact").is_(ckpt_output_col, "null").execute()
    print(f"Processed: {total_processed} | Remaining: {remaining_result.count}")

print(f"\n✅ Stage {STAGE} generation complete: {total_processed} records")

In [ ]:
# Cell 11: Simple evaluation - compare with teacher (sevenb)
from difflib import SequenceMatcher

def similarity_score(a: str, b: str) -> float:
    """Simple text similarity 0-1"""
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

print(f"\n📊 Evaluating Stage {STAGE} outputs...")

# Sample 500 records for evaluation
eval_result = supabase.table("modelcomp_50k") \
    .select(f"id, sevenb, student_output, {ckpt_output_col}") \
    .not_.is_(ckpt_output_col, "null") \
    .limit(500) \
    .execute()

scores = []
base_scores = []

for record in tqdm(eval_result.data, desc="Scoring"):
    teacher = record.get("sevenb") or ""
    base_student = record.get("student_output") or ""
    ckpt_student = record.get(ckpt_output_col) or ""
    
    # Score against teacher
    ckpt_score = similarity_score(ckpt_student, teacher)
    base_score = similarity_score(base_student, teacher)
    
    scores.append(ckpt_score)
    base_scores.append(base_score)
    
    # Update score column
    supabase.table("modelcomp_50k").update({
        f"score_ckpt{STAGE}": round(ckpt_score, 4)
    }).eq("id", record["id"]).execute()

avg_ckpt = sum(scores) / len(scores) if scores else 0
avg_base = sum(base_scores) / len(base_scores) if base_scores else 0
improvement = ((avg_ckpt - avg_base) / avg_base * 100) if avg_base > 0 else 0

print(f"\n{'='*50}")
print(f"📈 Stage {STAGE} Evaluation Results")
print(f"{'='*50}")
print(f"Base Student Similarity:    {avg_base:.4f}")
print(f"Stage {STAGE} Similarity:      {avg_ckpt:.4f}")
print(f"Improvement:                {improvement:+.2f}%")
print(f"{'='*50}")

In [ ]:
# Cell 12: Summary and next steps
print(f"\n🎉 STAGE {STAGE} COMPLETE!")
print(f"="*50)
print(f"Training records: {STAGE * RECORDS_PER_STAGE}")
print(f"Training time: {train_time/60:.1f} minutes")
print(f"Generation complete: {total_processed} records")
print(f"Similarity improvement: {improvement:+.2f}%")
print(f"\n📝 Next Steps:")
print(f"   1. Change STAGE = {STAGE + 1} in Cell 2")
print(f"   2. Runtime → Restart runtime")
print(f"   3. Run all cells again")
print(f"="*50)